# 01 — Prophet

Baseline forecasting TMA per stasiun memakai Prophet, dengan stasiun lain
sebagai regressor eksternal. Notebook ini berdiri sendiri: jalankan
`00_Preprocessing.ipynb` sekali dulu (datanya diambil dari BigQuery), lalu
Run All di sini.

Input-nya tabel `preprocessed_data_daily` + `preprocessing_metadata_daily`,
hasil `00_Preprocessing.ipynb` (dengan `GRANULARITY = "daily"`) atau
`make preprocess` (granularity `daily` di `config/preprocessing.yaml`).

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

import io
import json
import pickle
import numpy as np
import pandas as pd

PROJECT_ID = "thesis-506605"
DATASET    = "tables"
LOCATION   = "US"

PREPROCESSED_TABLE = f"{PROJECT_ID}.{DATASET}.preprocessed_data_daily"
METADATA_TABLE     = f"{PROJECT_ID}.{DATASET}.preprocessing_metadata_daily"

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

# --- Metadata: kolom kalender/stasiun/fitur + mapping nama kolom BigQuery ---
meta_row = client.query(f"""
    SELECT payload
    FROM `{METADATA_TABLE}`
    ORDER BY saved_at DESC
    LIMIT 1
""").to_dataframe().iloc[0]
metadata = json.loads(meta_row["payload"])
CALENDAR_COLS = metadata["CALENDAR_COLS"]
station_cols  = metadata["station_cols"]
feature_cols  = metadata["feature_cols"]
column_map    = metadata["column_map"]                 # station -> kolom BigQuery
reverse_column_map = {v: k for k, v in column_map.items()}
N_FEATURES = len(feature_cols)


# --- df_train / df_val / df_test dari tabel preprocessed_data --------------
def load_split(split):
    query = f"""
        SELECT *
        FROM `{PREPROCESSED_TABLE}`
        WHERE split = @split
        ORDER BY datetime
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("split", "STRING", split),
    ])
    df = client.query(query, job_config=job_config).to_dataframe()
    df = df.set_index("datetime").rename(columns=reverse_column_map)
    return df[feature_cols]


df_train = load_split("train")
df_val   = load_split("val")
df_test  = load_split("test")

print(f"Stations : {station_cols}")
print(f"Features : {N_FEATURES}")
print(f"Train/Val/Test: {len(df_train):,} / {len(df_val):,} / {len(df_test):,} rows")

## Konfigurasi BigQuery — Hasil Model

Tabel hasil untuk model ini (semua di dataset `tables`):

- `prophet_val_metrics_daily` — metrik validasi per (station, horizon, metric).
- `prophet_weights_daily` — bobot terlatih per stasiun, disimpan sebagai `BYTES`.
- `prophet_predictions_daily` — prediksi test-set per (station, horizon, datetime).
- `prophet_metrics_daily` — metrik test-set per (station, horizon, metric).

Bobot ditulis ke BigQuery segera setelah training (bukan ke disk), sehingga
langkah test-set evaluation di bawah bisa memuat ulang bobot dari BigQuery
saja bila objek model tidak lagi ada di memori (mis. notebook dibuka ulang).

In [ ]:
MODEL_NAME = "prophet"

VAL_METRICS_TABLE = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_val_metrics_daily"
WEIGHTS_TABLE     = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_weights_daily"
PREDICTIONS_TABLE = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_predictions_daily"
METRICS_TABLE     = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_metrics_daily"

WRITE_DISPOSITION = "WRITE_TRUNCATE"      # atau "WRITE_APPEND"


def load_df(df, table_id, schema):
    job_config = bigquery.LoadJobConfig(
        schema=schema,
        write_disposition=WRITE_DISPOSITION,
    )
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()   # tunggu selesai
    tbl = client.get_table(table_id)
    print(f"  {table_id}: {tbl.num_rows:,} rows")


print("Val metrics ->", VAL_METRICS_TABLE)
print("Weights     ->", WEIGHTS_TABLE)
print("Predictions ->", PREDICTIONS_TABLE)
print("Metrics     ->", METRICS_TABLE)

In [ ]:
# pip install prophet --quiet
from prophet import Prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

HORIZONS = [1, 3, 7]  # hari ke depan

In [ ]:
def make_prophet_df(df, target_station, other_stations):
    """Konversi DataFrame ke format Prophet (ds, y, + regressors)."""
    out = df[[target_station] + other_stations].copy()
    out = out.reset_index().rename(columns={
        "datetime": "ds",
        target_station: "y"
    })
    return out


def build_and_train_prophet(df_train_p, other_stations):
    """Fit Prophet dengan semua stasiun lain sebagai regressor."""
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,  # data harian: tidak ada pola intra-hari
        seasonality_mode="additive",
    )
    for station in other_stations:
        m.add_regressor(station)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m.fit(df_train_p)
    return m


def predict_horizon(model, df_future_p, horizon):
    """
    Prediksi h hari ke depan menggunakan nilai aktual sebagai regressor.
    Karena Prophet tidak native multi-step, kita shift target h langkah
    dan predict satu titik sekaligus (direct forecasting strategy).
    """
    df_shifted = df_future_p.copy()
    df_shifted["ds"] = df_shifted["ds"] + pd.Timedelta(days=horizon)
    forecast = model.predict(df_shifted)
    return forecast[["ds", "yhat"]].set_index("ds")["yhat"]


def compute_metrics(y_true, y_pred):
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)

    # sMAPE \u2014 symmetric, aman untuk nilai mendekati 0
    smape = np.mean(
        2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)
    ) * 100

    # NSE \u2014 Nash-Sutcliffe Efficiency
    nse = 1 - (np.sum((y_true - y_pred) ** 2) /
               np.sum((y_true - np.mean(y_true)) ** 2 + 1e-8))

    return {"RMSE": rmse, "MAE": mae, "sMAPE": smape, "NSE": nse}

In [ ]:
results = {}
models  = {}

for target in station_cols:
    other_stations = [s for s in station_cols if s != target]
    print(f"\n{'='*50}")
    print(f"Target: {target}")
    print(f"{'='*50}")

    df_train_p = make_prophet_df(df_train, target, other_stations)
    df_val_p   = make_prophet_df(df_val,   target, other_stations)

    model = build_and_train_prophet(df_train_p, other_stations)
    models[target] = model

    results[target] = {}

    for h in HORIZONS:
        preds = predict_horizon(model, df_val_p, horizon=h)

        y_true_index = df_val.index + pd.Timedelta(days=h)
        y_true = df_val[target].reindex(y_true_index).values
        y_pred = preds.reindex(y_true_index).values

        metrics = compute_metrics(y_true, y_pred)
        results[target][f"h+{h}"] = metrics

        print(f"  h+{h:>2}d \u2192 RMSE={metrics['RMSE']:.4f} | "
              f"MAE={metrics['MAE']:.4f} | "
              f"sMAPE={metrics['sMAPE']:.2f}% | "
              f"NSE={metrics['NSE']:.4f}")

In [ ]:
weights_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("weights", "BYTES", mode="REQUIRED"),
    bigquery.SchemaField("n_bytes", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("saved_at", "TIMESTAMP", mode="REQUIRED"),
]
val_metric_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("metric", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("value", "FLOAT", mode="NULLABLE"),
]

saved_at = pd.Timestamp.now(tz="UTC").tz_localize(None)

weight_rows = []
val_metric_rows = []
for station, model in models.items():
    blob = pickle.dumps(model)
    weight_rows.append({
        "model": MODEL_NAME, "station": station, "horizon": None,
        "weights": blob, "n_bytes": len(blob), "saved_at": saved_at,
    })
    for hkey, metrics in results[station].items():
        h = int(hkey.replace("h+", ""))
        for metric_name, value in metrics.items():
            val_metric_rows.append({
                "model": MODEL_NAME, "station": station, "horizon": h,
                "metric": metric_name, "value": float(value),
            })

weights_df = pd.DataFrame(weight_rows)
val_metrics_df = pd.DataFrame(val_metric_rows)
total_mb = weights_df["n_bytes"].sum() / 1e6
print(f"Menyiapkan {len(weights_df)} baris bobot ({total_mb:.2f} MB total).")

load_df(weights_df, WEIGHTS_TABLE, weights_schema)
load_df(val_metrics_df, VAL_METRICS_TABLE, val_metric_schema)
print(f"\n\u2714 Bobot + metrik validasi '{MODEL_NAME}' tersimpan di BigQuery.")

## Test Set Evaluation

Prophet dievaluasi pada test set (periode paling akhir, belum pernah
disentuh) memakai strategi direct forecasting yang sama seperti val set.

In [ ]:
def _load_weights_from_bq(station):
    query = f"""
        SELECT weights
        FROM `{WEIGHTS_TABLE}`
        WHERE station = @station
        ORDER BY saved_at DESC
        LIMIT 1
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("station", "STRING", station),
    ])
    row = client.query(query, job_config=job_config).to_dataframe()
    if row.empty:
        raise RuntimeError(f"Bobot Prophet utk '{station}' tidak ditemukan di {WEIGHTS_TABLE}.")
    return row.iloc[0]["weights"]


def predict_test_prophet(target, h):
    if target in models:
        model = models[target]
    else:
        # rebuild dari BigQuery (mis. notebook baru dibuka ulang)
        model = pickle.loads(_load_weights_from_bq(target))

    other = [s for s in station_cols if s != target]
    df_test_p = make_prophet_df(df_test, target, other)
    preds = predict_horizon(model, df_test_p, horizon=h)

    y_true_index = df_test.index + pd.Timedelta(days=h)
    y_true = df_test[target].reindex(y_true_index).values
    y_pred = preds.reindex(y_true_index).values
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    return y_true_index[mask], y_true[mask], y_pred[mask]


test_results = {}
test_predictions = []   # baris panjang utk BigQuery

for target in station_cols:
    test_results[target] = {}
    for h in HORIZONS:
        idx, y_true, y_pred = predict_test_prophet(target, h)
        metrics = compute_metrics(y_true, y_pred)
        test_results[target][f"h+{h}"] = metrics

        for ts, yt, yp in zip(idx, y_true, y_pred):
            test_predictions.append({
                "model": "prophet",
                "station": target,
                "horizon": int(h),
                "target_datetime": pd.Timestamp(ts),
                "y_true": float(yt),
                "y_pred": float(yp),
            })

    row = " | ".join(
        f"h+{h}: NSE={test_results[target][f'h+{h}']['NSE']:.3f}"
        for h in HORIZONS
    )
    print(f"{target:<20} {row}")

test_pred_df = pd.DataFrame(test_predictions)
print(f"\nTest predictions: {test_pred_df.shape[0]:,} rows.")
test_pred_df.head()

In [ ]:
metric_rows = []
for station, horizons in test_results.items():
    for hkey, metrics in horizons.items():
        for metric_name, value in metrics.items():
            metric_rows.append({
                "model": "prophet", "station": station,
                "horizon": int(hkey.replace("h+", "")),
                "metric": metric_name, "value": float(value),
            })
test_metrics_df = pd.DataFrame(metric_rows)
print(f"Test metrics rows: {len(test_metrics_df)}")

## Write Results to BigQuery

Prediksi dan metrik test-set model ini ditulis ke BigQuery. Autentikasi
mengikuti mekanisme standar Google Cloud (Application Default Credentials,
mis. kredensial bawaan Colab). Mode tulis default `WRITE_TRUNCATE` (ganti
seluruh isi tabel); ubah `WRITE_DISPOSITION` di atas ke `WRITE_APPEND` bila
ingin menumpuk hasil beberapa run.

In [ ]:
pred_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("target_datetime", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("y_true", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("y_pred", "FLOAT", mode="NULLABLE"),
]
metric_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("metric", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("value", "FLOAT", mode="NULLABLE"),
]

# target_datetime -> pastikan tz-naive UTC agar konsisten di BQ
_pred = test_pred_df.copy()
_pred["target_datetime"] = pd.to_datetime(_pred["target_datetime"])

print("Uploading to BigQuery ...")
load_df(_pred, PREDICTIONS_TABLE, pred_schema)
load_df(test_metrics_df, METRICS_TABLE, metric_schema)
print("\n\u2714 BigQuery write selesai.")